In [0]:
# Read the tables and choose the training cutoff.
# Leave as_of blank for the current time. A scheduled run will then use its run time.
dbutils.widgets.text("as_of", "")
as_of_input = dbutils.widgets.get("as_of").strip()

from datetime import datetime, timezone
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ByteType, ShortType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

NAMESPACE = "ml_projects.fraud_project"
MODEL_NAME = f"{NAMESPACE}.fraud_model"

AS_OF = as_of_input or datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
print(f"Training cutoff: {AS_OF}")

applications = spark.table(f"{NAMESPACE}.applications")
labels = spark.table(f"{NAMESPACE}.labels")

# Use records submitted before the cutoff and labels available before the cutoff.
training_sdf = (
    applications
    .filter(F.col("application_timestamp") < F.to_timestamp(F.lit(AS_OF)))
    .join(
        labels.filter(
            F.col("label_available_at") <= F.to_timestamp(F.lit(AS_OF))
        ),
        on="application_id",
        how="inner"
    )
)

# Choose numeric application fields as model inputs.
excluded_columns = {
    "application_id",
    "application_timestamp",
    "fraud_label",
    "label_available_at",
}

numeric_types = (
    ByteType, ShortType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType
)

feature_columns = [
    field.name
    for field in applications.schema.fields
    if field.name not in excluded_columns
    and isinstance(field.dataType, numeric_types)
]

if not feature_columns:
    raise ValueError("No numeric application feature columns were found.")

training_pdf = training_sdf.select(
    *feature_columns,
    "fraud_label"
).toPandas()

training_pdf = training_pdf.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["fraud_label"]
)

X = training_pdf[feature_columns].astype(float)
y = training_pdf["fraud_label"].astype(int)

if y.nunique() != 2:
    raise ValueError(
        f"Training data must contain both classes; found labels: {sorted(y.unique())}"
    )

print(f"Training rows: {len(training_pdf)}")
print(f"Fraud rows: {int(y.sum())}")
print(f"Features: {feature_columns}")

# Hold out data to estimate how well this candidate performs.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

estimator = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

estimator.fit(X_train, y_train)

probabilities = estimator.predict_proba(X_test)[:, 1]
predicted_labels = (probabilities >= 0.50).astype(int)

metrics = {
    "roc_auc": float(roc_auc_score(y_test, probabilities)),
    "precision_at_0_50": float(precision_score(
        y_test, predicted_labels, zero_division=0
    )),
    "recall_at_0_50": float(recall_score(
        y_test, predicted_labels, zero_division=0
    )),
}

print("Candidate metrics:", metrics)

# The wrapper keeps the endpoint response fields used by the simulation.
class FraudScoringModel(mlflow.pyfunc.PythonModel):
    def __init__(self, fitted_estimator, input_features, model_build_id):
        self.fitted_estimator = fitted_estimator
        self.input_features = input_features
        self.model_build_id = model_build_id

    def predict(self, context, model_input, params=None):
        model_input = model_input[self.input_features].copy()
        scores = self.fitted_estimator.predict_proba(model_input)[:, 1]

        return pd.DataFrame({
            "fraud_probability": scores,
            "review_required": (scores >= 0.50),
            "model_build_id": self.model_build_id,
        })

sample_input = X_test.head(3)
sample_output = pd.DataFrame({
    "fraud_probability": estimator.predict_proba(sample_input)[:, 1],
    "review_required": estimator.predict_proba(sample_input)[:, 1] >= 0.50,
    "model_build_id": [f"candidate_as_of_{AS_OF}"] * len(sample_input),
})
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="fraud_retraining_candidate") as run:
    mlflow.log_param("as_of", AS_OF)
    mlflow.log_param("feature_columns", ",".join(feature_columns))
    mlflow.log_param("training_rows", len(training_pdf))
    mlflow.log_metrics(metrics)

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FraudScoringModel(
            estimator,
            feature_columns,
            f"candidate_{run.info.run_id[:8]}",
        ),
        input_example=sample_input,
        signature=signature,
        registered_model_name=MODEL_NAME,
    )

client = MlflowClient()
registered_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
candidate_version = max(int(v.version) for v in registered_versions)

print(f"Registered candidate: {MODEL_NAME}, version {candidate_version}")
print(f"Metrics: {metrics}")

In [0]:
import pandas as pd
import mlflow
from sklearn.metrics import roc_auc_score, precision_score, recall_score

MODEL_NAME = "ml_projects.fraud_project.fraud_model"
CURRENT_VERSION = "3"

current_model = mlflow.pyfunc.load_model(
    f"models:/{MODEL_NAME}/{CURRENT_VERSION}"
)

model_info = mlflow.models.get_model_info(
    f"models:/{MODEL_NAME}/{CURRENT_VERSION}"
)
expected_columns = model_info.signature.inputs.input_names()

# The registered model requires application_id, but it isn't a model feature
# in X_test. Add temporary IDs to satisfy its input signature.
current_input = X_test.copy()
current_input.insert(
    0,
    "application_id",
    [f"VALIDATION-{i}" for i in range(len(current_input))]
)

# Send exactly the columns and order expected by version 3.
current_input = current_input[expected_columns]

current_output = current_model.predict(current_input)
current_probabilities = (
    current_output["fraud_probability"].astype(float).to_numpy()
)
candidate_probabilities = estimator.predict_proba(X_test)[:, 1]

def calculate_metrics(y_true, probabilities):
    predicted = (probabilities >= 0.50).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "Precision": precision_score(y_true, predicted, zero_division=0),
        "Recall": recall_score(y_true, predicted, zero_division=0),
    }

comparison = pd.DataFrame({
    "Currently serving version 3":
        calculate_metrics(y_test, current_probabilities),
    "New candidate":
        calculate_metrics(y_test, candidate_probabilities),
})

display(comparison)

In [0]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score

threshold_rows = []

for model_name, probabilities in [
    ("Serving v3", current_probabilities),
    ("Candidate", candidate_probabilities),
]:
    for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
        flagged = (probabilities >= threshold).astype(int)

        threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(
                y_test, flagged, zero_division=0
            ),
            "recall": recall_score(
                y_test, flagged, zero_division=0
            ),
            "applications_flagged_for_review": int(flagged.sum()),
            "review_rate": float(flagged.mean()),
        })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

In [0]:
dbutils.notebook.exit(str(candidate_version))